In [3]:
#
# Wichtigste Änderungen gegenüber 04_Modeling:
#   - Log-Target: trainiert auf log1p(sales), Vorhersage mit expm1
#   - Keine Leakage-Features (log_sales / sales_per_transaction entfernt)
#   - Nur Lags >= 16 (aus 05_feature_engineering_v2)
#   - Walk-Forward Cross-Validation (3 Folds)
#   - Finales Modell auf vollem Train, Submission vorbereiten
# 

In [7]:

from pathlib import Path
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.xgboost

In [8]:
BASE_DIR      = Path(r"C:\Users\nelid\Documents\Kaggle Competitions\Store Sales Competition\store-sales")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

In [10]:
# Daten laden
train_part = pd.read_csv(PROCESSED_DIR / "train_lagged_train_v2.csv", parse_dates=["date"])
val_part   = pd.read_csv(PROCESSED_DIR / "train_lagged_val_v2.csv",   parse_dates=["date"])
test_fe    = pd.read_csv(PROCESSED_DIR / "test_lagged_v2.csv",        parse_dates=["date"])


C:\Users\nelid\AppData\Local\Temp\ipykernel_8608\3049760032.py:2: DtypeWarning: Columns (48) have mixed types. Specify dtype option on import or set low_memory=False.
  train_part = pd.read_csv(PROCESSED_DIR / "train_lagged_train_v2.csv", parse_dates=["date"])
C:\Users\nelid\AppData\Local\Temp\ipykernel_8608\3049760032.py:4: DtypeWarning: Columns (48) have mixed types. Specify dtype option on import or set low_memory=False.
  test_fe    = pd.read_csv(PROCESSED_DIR / "test_lagged_v2.csv",        parse_dates=["date"])


In [11]:
# Feature-Spalten definieren
EXCLUDE  = ["id", "date", "sales"]
cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster", "holiday_names"]
num_cols = [c for c in train_part.columns if c not in cat_cols + EXCLUDE]
feature_cols = cat_cols + num_cols

print(f"Anzahl Features: {len(feature_cols)}")



Anzahl Features: 117


In [12]:
# Encoding
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

for df in [train_part, val_part, test_fe]:
    for c in cat_cols:
        df[c] = df[c].astype(str)

train_part[cat_cols] = enc.fit_transform(train_part[cat_cols])
val_part[cat_cols]   = enc.transform(val_part[cat_cols])
test_fe[cat_cols]    = enc.transform(test_fe[cat_cols])


In [13]:
# Metriken

def rmsle(y_true, y_pred):
    y_true_c = np.maximum(y_true, 0)
    y_pred_c = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred_c) - np.log1p(y_true_c)) ** 2))

def mape(y_true, y_pred):
    y_true_c = np.where(y_true == 0, 1e-6, y_true)
    return np.mean(np.abs(y_true - y_pred) / y_true_c) * 100


In [16]:
# LOG-TARGET: Modell auf log1p(sales) trainieren

# Schritt 1: ERST alle DataFrames bereinigen (vor dem Extrahieren der Feature-Matrizen)
def fix_dtypes(df, feature_cols):
    df = df.copy()
    for col in feature_cols:
        if col in df.columns and df[col].dtype not in ["float64", "float32", "int64", "int32", "int8", "bool"]:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)
    return df

train_part = fix_dtypes(train_part, feature_cols)
val_part   = fix_dtypes(val_part,   feature_cols)
test_fe    = fix_dtypes(test_fe,    feature_cols)

# Sicherheits-Check
bad_cols = [c for c in feature_cols if train_part[c].dtype == object]
if bad_cols:
    print("Noch object-Spalten:", bad_cols)
else:
    print("Alle Feature-Spalten sind numerisch.")




Alle Feature-Spalten sind numerisch.


In [17]:
# Schritt 2: DANN extrahieren
y_train  = np.log1p(train_part["sales"].clip(lower=0))
y_val    = val_part["sales"].clip(lower=0)
test_ids = test_fe["id"]

X_train = train_part[feature_cols]
X_val   = val_part[feature_cols]
X_test  = test_fe[feature_cols]

# Schritt 3: DMatrix bauen
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=np.log1p(y_val))
dtest  = xgb.DMatrix(X_test)

print("DMatrix erfolgreich erstellt.")
print(f"Train: {dtrain.num_row()} Zeilen, {dtrain.num_col()} Features")
print(f"Val:   {dval.num_row()} Zeilen")
print(f"Test:  {dtest.num_row()} Zeilen")


DMatrix erfolgreich erstellt.
Train: 2974158 Zeilen, 117 Features
Val:   26730 Zeilen
Test:  28512 Zeilen


In [18]:
#  Modell trainieren (beste Parameter aus 04_Modeling als Ausgangspunkt)
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("store-sales-sarima")

params = {
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "eta":              0.05,
    "max_depth":        8,
    "min_child_weight": 3,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "lambda":           1.0,
    "alpha":            0.1,
    "tree_method":      "hist",
}

with mlflow.start_run(run_name="xgb_v2_log_target"):
    mlflow.log_params(params)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=50,
        verbose_eval=100,
    )

   # Vorhersage: expm1 ruecktransformieren
    y_pred_log = model.predict(dval)
    y_pred     = np.expm1(y_pred_log).clip(min=0)

    rmsle_val  = rmsle(y_val.values, y_pred)
    mape_val   = mape(y_val.values, y_pred)
    rmse_val   = np.sqrt(mean_squared_error(y_val, y_pred))

    print(f"RMSLE: {rmsle_val:.5f}")
    print(f"MAPE:  {mape_val:.2f}%")
    print(f"RMSE:  {rmse_val:.2f}")

    mlflow.log_metric("rmsle_val", rmsle_val)
    mlflow.log_metric("mape_val",  mape_val)
    mlflow.log_metric("rmse_val",  rmse_val)
    mlflow.log_metric("best_iteration", model.best_iteration)

    # Per Store-Family RMSLE
    val_part["pred"] = y_pred
    group_cols = ["store_nbr", "family"]
    per_series = (
        val_part
        .groupby(group_cols)
        .apply(lambda g: pd.Series({
            "rmsle": rmsle(g["sales"].values, g["pred"].values),
            "n":     len(g),
        }))
        .reset_index()
    )
    print(f"RMSLE Mittelwert per Serie: {per_series['rmsle'].mean():.5f}")
    mlflow.log_metric("rmsle_series_mean", per_series["rmsle"].mean())

    mlflow.xgboost.log_model(model, artifact_path="model")
    best_n_rounds = model.best_iteration + 1



[0]	train-rmse:2.56814	valid-rmse:2.49354
[100]	train-rmse:0.41919	valid-rmse:0.40653
[200]	train-rmse:0.39108	valid-rmse:0.39516
[300]	train-rmse:0.37489	valid-rmse:0.39098
[400]	train-rmse:0.36657	valid-rmse:0.38861
[500]	train-rmse:0.36128	valid-rmse:0.38672
[574]	train-rmse:0.35796	valid-rmse:0.38672
RMSLE: 0.38671
MAPE:  5985260.34%
RMSE:  209.58


C:\Users\nelid\AppData\Local\Temp\ipykernel_8608\3136680394.py:53: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
2026/02/27 21:11:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSLE Mittelwert per Serie: 0.32238
🏃 View run xgb_v2_log_target at: http://127.0.0.1:5000/#/experiments/1/runs/95c465ad10a54d3ba68fcbef23dd6b43
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:
### Diagnose

# Wie viele NaN-Werte haben die wichtigsten Lag-Features im Val-Set?
lag_cols = [c for c in val_part.columns if "sales_lag" in c or "sales_rmean" in c]
nan_rates = val_part[lag_cols].isna().mean().sort_values(ascending=False)
print("NaN-Rate der Lag-Features im Val-Set:")
print(nan_rates.head(20))



NaN-Rate der Lag-Features im Val-Set:
sales_lag_16               0.0
sales_lag_21               0.0
sales_lag_28               0.0
sales_lag_35               0.0
sales_lag_42               0.0
sales_lag_56               0.0
sales_lag_90               0.0
sales_rmean_lag16_win7     0.0
sales_rmean_lag16_win14    0.0
sales_rmean_lag16_win28    0.0
sales_rmean_lag16_win56    0.0
sales_rmean_lag16_win90    0.0
sales_rmean_lag21_win7     0.0
sales_rmean_lag21_win14    0.0
sales_rmean_lag21_win28    0.0
sales_rmean_lag21_win56    0.0
sales_rmean_lag21_win90    0.0
sales_rmean_lag28_win7     0.0
sales_rmean_lag28_win14    0.0
sales_rmean_lag28_win28    0.0
dtype: float64


In [20]:
# Wie viele NaN-Werte im Train-Set?
nan_rates_train = train_part[lag_cols].isna().mean().sort_values(ascending=False)
print("\nNaN-Rate der Lag-Features im Train-Set:")
print(nan_rates_train.head(20))





NaN-Rate der Lag-Features im Train-Set:
sales_lag_90               0.053925
sales_rmean_lag90_win90    0.053925
sales_rmean_lag90_win7     0.053925
sales_rmean_lag90_win28    0.053925
sales_rmean_lag90_win56    0.053925
sales_rmean_lag90_win14    0.053925
sales_rmean_lag56_win90    0.033553
sales_rmean_lag56_win7     0.033553
sales_rmean_lag56_win14    0.033553
sales_lag_56               0.033553
sales_rmean_lag56_win56    0.033553
sales_rmean_lag56_win28    0.033553
sales_rmean_lag42_win56    0.025165
sales_rmean_lag42_win90    0.025165
sales_rmean_lag42_win7     0.025165
sales_rmean_lag42_win14    0.025165
sales_rmean_lag42_win28    0.025165
sales_lag_42               0.025165
sales_rmean_lag35_win56    0.020971
sales_rmean_lag35_win28    0.020971
dtype: float64


In [21]:
# Wie viel Prozent der Val-Zeilen haben NaN in sales_lag_16?
print("\nNaN in sales_lag_16 (Val):", val_part["sales_lag_16"].isna().mean())
print("NaN in sales_lag_16 (Train):", train_part["sales_lag_16"].isna().mean())




NaN in sales_lag_16 (Val): 0.0
NaN in sales_lag_16 (Train): 0.009586578789694428


In [22]:
# Feature Importances anschauen
import pandas as pd
score = model.get_score(importance_type="gain")
imp_df = (
    pd.DataFrame([(k, v) for k, v in score.items()], columns=["feature", "importance"])
    .sort_values("importance", ascending=False)
)
print("\nTop 20 Features:")
print(imp_df.head(20))


Top 20 Features:
                       feature    importance
67      sales_rmean_lag16_win7  38949.687500
61                sales_lag_21  10905.662109
71     sales_rmean_lag16_win90   9248.777344
68     sales_rmean_lag16_win14   9183.166992
70     sales_rmean_lag16_win56   4437.437500
69     sales_rmean_lag16_win28   1357.275391
65                sales_lag_56   1036.704468
91     sales_rmean_lag42_win90    639.386353
89     sales_rmean_lag42_win28    634.188354
60                sales_lag_16    590.272461
15                     quarter    548.549194
109  store_family_median_sales    507.727325
76     sales_rmean_lag21_win90    476.975159
113     store_family_vs_global    413.016754
80     sales_rmean_lag28_win56    384.395477
108    store_family_mean_sales    375.632385
62                sales_lag_28    334.951324
74     sales_rmean_lag21_win28    299.754395
64                sales_lag_42    277.992035
7                  onpromotion    274.292267


In [23]:
print("Train:", train_part["date"].min(), "->", train_part["date"].max())
print("Val:  ", val_part["date"].min(),   "->", val_part["date"].max())
print("Train-Zeilen:", len(train_part), "| Val-Zeilen:", len(val_part))
print("Best iteration:", model.best_iteration)


Train: 2013-01-01 00:00:00 -> 2017-07-31 00:00:00
Val:   2017-08-01 00:00:00 -> 2017-08-15 00:00:00
Train-Zeilen: 2974158 | Val-Zeilen: 26730
Best iteration: 524


In [ ]:
# Finales Modell auf vollem Train (Train+Val) trainieren

full_train = pd.concat([train_part, val_part], axis=0)
X_full     = full_train[feature_cols]
y_full     = np.log1p(full_train["sales"].clip(lower=0))
dtrain_full = xgb.DMatrix(X_full, label=y_full)

with mlflow.start_run(run_name="xgb_v2_final_full_train"):
    mlflow.log_params(params)
    mlflow.log_param("num_boost_round", best_n_rounds)

    final_model = xgb.train(
        params,
        dtrain_full,
        num_boost_round=best_n_rounds,
        verbose_eval=100,
    )

    mlflow.xgboost.log_model(final_model, artifact_path="model")



In [ ]:
# Submission erstellen


test_pred = np.expm1(final_model.predict(dtest)).clip(min=0)

submission = pd.DataFrame({"id": test_ids, "sales": test_pred})
submission.to_csv(PROCESSED_DIR / "submission_xgb_v2.csv", index=False